# Lab 20 — Cyber Threat Intelligence Analytics

*AI Cybersecurity Masterclass · Lab 20 · AI60secLab*

**Research question:** What can we learn by analyzing ATT&CK relationships between adversary groups and techniques?

Instead of treating CTI as prose, we transform STIX relationships into structured evidence.

In [ ]:
# Setup: works in Google Colab and locally. No uploads needed.
# In Colab this clones the course repo, which already includes the UNSW-NB15 data.
from pathlib import Path
import sys, os

REPO_URL = "https://github.com/Jacquelinepersha/ai-cybersecurity-public-labs.git"
REPO_NAME = "ai-cybersecurity-public-labs"

try:
    import google.colab  # noqa: F401
    if not Path(REPO_NAME).exists():
        !git clone --depth 1 {REPO_URL}
    os.chdir(REPO_NAME)
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "raw"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data folder :", DATA_DIR)

In [ ]:
import matplotlib.pyplot as plt

from src.mitre_data import (
    ATTACK_ENTERPRISE_URL,
    load_remote_or_local,
)
from src.cti import (
    group_technique_edges,
    technique_usage_frequency,
    group_profile,
    tactic_coverage_for_group,
)

local_fallback = DATA_DIR / "enterprise-attack.json"

attack_bundle = load_remote_or_local(
    ATTACK_ENTERPRISE_URL,
    local_path=local_fallback,
)

edges = group_technique_edges(attack_bundle)
frequency = technique_usage_frequency(attack_bundle)

print("Group-technique relationships:", len(edges))
display(frequency.head(20))

In [ ]:
top = frequency.head(20).sort_values("group_count")

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(
    top["technique_name"],
    top["group_count"],
)
ax.set_xlabel("Number of ATT&CK groups")
ax.set_title("Most widely mapped techniques across groups")
plt.tight_layout()
plt.show()

In [ ]:
group_query = "APT29"

profile = group_profile(
    attack_bundle,
    group_query,
)

print("Mapped techniques:", len(profile))
display(profile.head(30))

coverage = tactic_coverage_for_group(profile)
display(coverage)

## Intelligence interpretation

- Which tactics have the largest mapped coverage for the selected group?
- Which techniques are common across many groups?
- What telemetry would be required to observe these behaviors?
- Why should ATT&CK mappings organize evidence rather than be treated as automatic attribution?

**Takeaway:** CTI becomes more useful when relationships are queryable and tied to observable evidence.